# Tutorial 05: Delegation Loop End-to-End

This notebook runs AWP's **delegation loop engine** end-to-end with a real LLM.
The delegation loop is the orchestration engine for **A2-A4 autonomy levels** --
it implements a dynamic manager-worker pattern where the manager dispatches tasks
to ephemeral workers, collects results, and decides the next step.

**Prerequisites:**
- `pip install -e "reference/python/[data]"`
- An LLM API key (OpenRouter, Ollama, or any OpenAI-compatible endpoint)

## 1. Provider Setup

Choose your LLM provider below. Supported options:
- **Ollama** -- local, no API key needed (install from ollama.com)
- **OpenRouter** -- cloud, requires `OPENROUTER_API_KEY`
- **Custom API** -- any OpenAI-compatible endpoint, requires `LLM_API_KEY` + `LLM_BASE_URL`

In [1]:
# ============================================================
# Provider Selection — choose ONE of: "ollama", "openrouter", "custom"
# ============================================================
PROVIDER = "openrouter"  # <-- change this

# --- Ollama (local) -------------------------------------------
OLLAMA_MODEL = "qwen3:1.7b"            # any model you've pulled
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# --- OpenRouter (cloud) ---------------------------------------
OPENROUTER_API_KEY = ""  # paste your key or set env var OPENROUTER_API_KEY
OPENROUTER_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

# --- Custom OpenAI-compatible API -----------------------------
CUSTOM_API_KEY = ""                     # paste your key or set env var LLM_API_KEY
CUSTOM_BASE_URL = ""                    # e.g. "https://api.openai.com/v1"
CUSTOM_MODEL = ""                       # e.g. "gpt-4o"

# ==============================================================
# DO NOT EDIT BELOW — wires up the selected provider
# ==============================================================
import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
    print(f"Using Ollama  model={OLLAMA_MODEL}  url={OLLAMA_BASE_URL}")

elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
    print(f"Using OpenRouter  model={OPENROUTER_MODEL}")

elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key:
        raise ValueError("Set CUSTOM_API_KEY above or LLM_API_KEY as an environment variable")
    if not url:
        raise ValueError("Set CUSTOM_BASE_URL above or LLM_BASE_URL as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
    print(f"Using custom API  model={CUSTOM_MODEL}  url={url}")

else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Use 'ollama', 'openrouter', or 'custom'.")

os.environ["LLM_MODEL"] = MODEL
print(f"LLM_MODEL={MODEL}")
print(f"LLM_BASE_URL={os.environ['LLM_BASE_URL']}")

Using OpenRouter  model=openrouter/google/gemini-2.5-flash
LLM_MODEL=openrouter/google/gemini-2.5-flash
LLM_BASE_URL=https://openrouter.ai/api/v1


## 2. What is the Delegation Loop?

The delegation loop is AWP's orchestration engine for **A2-A4 autonomy levels**.
Unlike the DAG engine (A0-A1), which follows a fixed dependency graph, the delegation
loop is **dynamic** -- the manager decides at runtime what to do next.

### How it works

```
                    +-------------------+
                    |     MANAGER       |
                    |  (receives task)  |
                    +--------+----------+
                             |
                    Decides: DELEGATE / COMPLETE / FAIL
                             |
              +--------------+--------------+
              |              |              |
        +-----v-----+ +-----v-----+ +-----v-----+
        |  Worker A  | |  Worker B  | |  Worker C  |
        | (ephemeral)| | (ephemeral)| | (ephemeral)|
        +-----+------+ +-----+------+ +-----+------+
              |              |              |
              +------+-------+------+-------+
                     |              |
              Results collected     |
                     |              |
              +------v--------------v------+
              |  MANAGER reviews results   |
              |  Next iteration or COMPLETE |
              +----------------------------+
```

### Key features

- **Budget system**: Hard limits on loops, workers, tokens, wall time, and recursion depth.
  The manager cannot override these -- they guarantee termination.
- **Stall detection**: If confidence does not improve over a sliding window, the loop
  warns and then stops -- preventing infinite spinning.
- **Validation gates**: Deterministic validation runs on every result; optional LLM-based
  semantic validation when confidence is below threshold.
- **Workers are ephemeral**: Each worker is configured at runtime by the manager's delegation
  envelope (instructions, tools, skills). No static YAML needed.

### When to use

| Autonomy Level | Engine | Use Case |
|---|---|---|
| A0-A1 | DAG | Fixed pipelines, known steps |
| A2 | Delegation Loop | Manager directs workers with known skills |
| A3 | Delegation Loop | Workers create tools dynamically |
| A4 | Delegation Loop | Self-organizing recursive delegation |

## 3. Using AgentWorkflow (Simple API)

`AgentWorkflow` is a high-level wrapper around the delegation loop.
You pass inputs + a task description, and it handles all the wiring
(workspace setup, manager prompt, tool registry, budget config).

In [2]:
from awp.data import AgentWorkflow

result = AgentWorkflow(
    inputs={"numbers": [10, 20, 30, 40, 50]},
    task="Calculate the mean, median, and standard deviation of the numbers. Return the results as JSON.",
    model=MODEL,
    max_loops=3,
    max_wall_time=120,
    max_total_tokens=100_000,
    output_dir="./output_delegation_simple",
    verbose=True,
).run()

print(f"Status: {result['status']}")
print(f"Result: {result['result']}")
print(f"Metadata: {result['metadata']}")

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_simple


INFO:awp.data.inputs:Input 'numbers': List -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_simple/workspace/inputs/numbers.json


INFO:awp.data.workflow:Starting delegation loop: task=Calculate the mean, median, and standard deviation of the numbers. Return the re


INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_03-16-47_d3539829] depth=0 starting: Calculate the mean, median, and standard deviation of the numbers. Return the re


INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===


...

ERROR:awp.runtime.agent:Agent manager LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:awp.runtime.delegation_loop_runner:Inline manager failed: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401



  AWP DELEGATION LOOP DEBUG REPORT
  Model:         openrouter/google/gemini-2.5-flash
  Worker model:  openrouter/google/gemini-2.5-flash
  Budget:        loops=3, workers=30, tokens=100,000, wall_time=120s, depth=5

  DELEGATION LOOP SUMMARY
  Duration:    0.5s
  Status:      fail
  Iterations:  1
  Workers:     0/30
  Budget:      66.7% remaining
  Files:       3 (1.4KB)
  Run dir:     /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_simple/workspace/runs/2026-03-28_03-16-47_d3539829
Status: error
Result: {'error': "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401", 'partial_result': {}, 'confidence': 0.0}
Metadata: {'loops': 1, 'tokens_used': 0, 'wall_time': 0.5, 'workers_spawned': 0, 'tool_calls': 0, 'workspace': '/home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_simple

In [3]:
# Inspect the result in detail
import json

print("=== Full Result ===")
print(json.dumps(result["result"], indent=2, default=str))
print()
print("=== Metadata ===")
print(json.dumps(result["metadata"], indent=2, default=str))
print()
print(f"Artifacts: {result['artifacts']}")

=== Full Result ===
{
  "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
  "partial_result": {},
  "confidence": 0.0
}

=== Metadata ===
{
  "loops": 1,
  "tokens_used": 0,
  "wall_time": 0.5,
  "workers_spawned": 0,
  "tool_calls": 0,
  "workspace": "/home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_simple"
}

Artifacts: []


## 4. AgentWorkflow with DataFrame

Pass a pandas DataFrame as input. `AgentWorkflow` automatically serializes it
to CSV in the workspace so the agent can read it.

In [4]:
import pandas as pd

df = pd.DataFrame({
    "product": ["Widget A", "Widget B", "Widget C"] * 10,
    "sales": [100 + i * 5 for i in range(30)],
    "month": list(range(1, 31)),
})

print(f"DataFrame shape: {df.shape}")
df.head(10)

DataFrame shape: (30, 3)


,product,sales,month
0,Widget A,100,1
1,Widget B,105,2
2,Widget C,110,3
3,Widget A,115,4
4,Widget B,120,5
5,Widget C,125,6
6,Widget A,130,7
7,Widget B,135,8
8,Widget C,140,9
9,Widget A,145,10


In [5]:
result_df = AgentWorkflow(
    inputs={"sales_data": df},
    task="Analyze the sales data: calculate total sales per product, find the best month, and identify trends.",
    model=MODEL,
    max_loops=5,
    max_wall_time=180,
    output_dir="./output_delegation_df",
).run()

print(f"Status: {result_df['status']}")
print(f"Loops used: {result_df['metadata']['loops']}")
print(f"Wall time: {result_df['metadata']['wall_time']}s")
print()
print(json.dumps(result_df["result"], indent=2, default=str))

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_df


INFO:awp.data.inputs:Input 'sales_data': DataFrame -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_df/workspace/inputs/sales_data.csv


INFO:awp.data.workflow:Starting delegation loop: task=Analyze the sales data: calculate total sales per product, find the best month, 


INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_03-16-48_4c4879fe] depth=0 starting: Analyze the sales data: calculate total sales per product, find the best month, 


INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===


...

ERROR:awp.runtime.agent:Agent manager LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:awp.runtime.delegation_loop_runner:Inline manager failed: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


Status: error
Loops used: 1
Wall time: 0.21s

{
  "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
  "partial_result": {},
  "confidence": 0.0
}


## 5. Budget Enforcement

The budget system guarantees that the delegation loop always terminates.
Hard limits include:

| Budget Parameter | What it limits |
|---|---|
| `max_loops` | Total manager iterations |
| `max_total_workers` | Total worker agents spawned |
| `max_total_tokens` | Total LLM tokens consumed |
| `max_wall_time` | Wall clock seconds |
| `max_tool_calls` | Total tool invocations |
| `max_depth` | Recursive delegation depth |

The manager **cannot override** these limits. Let's see what happens with a very tight budget:

In [6]:
result_budget = AgentWorkflow(
    inputs={"data": "test"},
    task="Write a long analysis of machine learning trends in 2025.",
    model=MODEL,
    max_loops=1,
    max_wall_time=30,
    max_total_tokens=5000,
    output_dir="./output_budget_test",
).run()

print(f"Status: {result_budget['status']}")
print(f"Loops used: {result_budget['metadata']['loops']}")
print(f"Tokens used: {result_budget['metadata']['tokens_used']}")
print(f"Wall time: {result_budget['metadata']['wall_time']}s")
print()
print("Result (may be partial or budget_exceeded):")
print(json.dumps(result_budget["result"], indent=2, default=str))

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_budget_test


INFO:awp.data.workflow:Starting delegation loop: task=Write a long analysis of machine learning trends in 2025.


INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_03-16-48_b1293dc3] depth=0 starting: Write a long analysis of machine learning trends in 2025.


INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


...

ERROR:awp.runtime.agent:Agent manager LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:awp.runtime.delegation_loop_runner:Inline manager failed: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


Status: error
Loops used: 1
Tokens used: 0
Wall time: 0.2s

Result (may be partial or budget_exceeded):
{
  "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
  "partial_result": {},
  "confidence": 0.0
}


## 6. Tool Creation and Code Mode

With `code_mode=True` and `tool_creation=True`, workers can:
- Execute Python code in a sandboxed environment (`code.execute`)
- Create new tools dynamically at runtime

This is what enables A3-A4 autonomy levels.

In [7]:
result_tools = AgentWorkflow(
    inputs={"values": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]},
    task="Create a custom tool that calculates fibonacci numbers, then use it to compute fib(10). Return the result.",
    model=MODEL,
    code_mode=True,
    tool_creation=True,
    max_loops=5,
    max_wall_time=120,
    output_dir="./output_tool_creation",
    verbose=True,
).run()

print(f"Status: {result_tools['status']}")
print(f"Loops used: {result_tools['metadata']['loops']}")
print(f"Workers spawned: {result_tools['metadata']['workers_spawned']}")
print()
print(json.dumps(result_tools["result"], indent=2, default=str))

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_tool_creation


INFO:awp.data.inputs:Input 'values': List -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_tool_creation/workspace/inputs/values.json


INFO:awp.data.workflow:Starting delegation loop: task=Create a custom tool that calculates fibonacci numbers, then use it to compute f


INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_03-16-48_8653d014] depth=0 starting: Create a custom tool that calculates fibonacci numbers, then use it to compute f


INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===


...

ERROR:awp.runtime.agent:Agent manager LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:awp.runtime.delegation_loop_runner:Inline manager failed: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401



  AWP DELEGATION LOOP DEBUG REPORT
  Model:         openrouter/google/gemini-2.5-flash
  Worker model:  openrouter/google/gemini-2.5-flash
  Budget:        loops=5, workers=30, tokens=500,000, wall_time=120s, depth=5

  DELEGATION LOOP SUMMARY
  Duration:    0.2s
  Status:      fail
  Iterations:  1
  Workers:     0/30
  Budget:      80.0% remaining
  Files:       3 (1.5KB)
  Run dir:     /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_tool_creation/workspace/runs/2026-03-28_03-16-48_8653d014
Status: error
Loops used: 1
Workers spawned: 0

{
  "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
  "partial_result": {},
  "confidence": 0.0
}


## 7. Inspecting Logs and Artifacts

The delegation loop writes detailed logs and artifacts to disk.
Each run creates a directory under `<output_dir>/workspace/runs/<run_id>/`
containing:

- `run_manifest.json` -- run configuration
- `RUN_SUMMARY.md` -- human-readable summary
- `iterations/NNN/` -- per-iteration manager decisions and worker results
- `artifacts/tools/` -- generated tool code and specs
- `artifacts/skills/` -- generated skill documents

In [8]:
from pathlib import Path

# Find the workspace from the last run
workspace = Path(result["metadata"]["workspace"])
print(f"Workspace: {workspace}")
print()

# List the top-level structure
for p in sorted(workspace.rglob("*")):
    if p.is_file():
        rel = p.relative_to(workspace)
        size = p.stat().st_size
        print(f"  {rel}  ({size} bytes)")

Workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_delegation_simple

  agents/manager/agent.awp.yaml  (344 bytes)
  agents/manager/system_prompt.md  (4428 bytes)
  workspace/input_manifest.json  (144 bytes)
  workspace/inputs/numbers.json  (32 bytes)
  workspace/runs/2026-03-28_03-16-47_d3539829/RUN_SUMMARY.md  (476 bytes)
  workspace/runs/2026-03-28_03-16-47_d3539829/run_completion.json  (508 bytes)
  workspace/runs/2026-03-28_03-16-47_d3539829/run_manifest.json  (498 bytes)


In [9]:
# Read the run manifest (if it exists)
runs_dir = workspace / "workspace" / "runs"
if not runs_dir.exists():
    runs_dir = workspace / "runs"

if runs_dir.exists():
    run_dirs = sorted(runs_dir.glob("*"))
    if run_dirs:
        latest_run = run_dirs[-1]
        print(f"Latest run: {latest_run.name}")
        print()

        # Show run manifest
        manifest_path = latest_run / "run_manifest.json"
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text())
            print("=== Run Manifest ===")
            print(json.dumps(manifest, indent=2, default=str))
            print()

        # Show iteration logs
        iter_dir = latest_run / "iterations"
        if iter_dir.exists():
            for idir in sorted(iter_dir.glob("*")):
                decision_file = idir / "manager_decision.json"
                if decision_file.exists():
                    decision = json.loads(decision_file.read_text())
                    print(f"=== Iteration {idir.name} ===")
                    print(json.dumps(decision, indent=2, default=str)[:1000])
                    print()
else:
    print("No runs directory found. The run may have been too short to log.")

Latest run: 2026-03-28_03-16-47_d3539829

=== Run Manifest ===
{
  "run_id": "2026-03-28_03-16-47_d3539829",
  "task": "Calculate the mean, median, and standard deviation of the numbers. Return the results as JSON.",
  "started": "2026-03-28T03:16:47.742847+00:00",
  "models": {
    "manager": "openrouter/google/gemini-2.5-flash",
    "worker": "openrouter/google/gemini-2.5-flash"
  },
  "budget": {
    "max_loops": 3,
    "max_total_workers": 30,
    "max_total_tokens": 100000,
    "max_wall_time": 120,
    "max_tool_calls": 100,
    "max_depth": 5
  }
}



In [10]:
# Check for generated artifacts (tools, skills)
artifacts_dir = None
if runs_dir.exists():
    run_dirs = sorted(runs_dir.glob("*"))
    if run_dirs:
        artifacts_dir = run_dirs[-1] / "artifacts"

if artifacts_dir and artifacts_dir.exists():
    print("=== Generated Artifacts ===")
    for p in sorted(artifacts_dir.rglob("*")):
        if p.is_file():
            rel = p.relative_to(artifacts_dir)
            print(f"  {rel}  ({p.stat().st_size} bytes)")
            # Show content of small files
            if p.stat().st_size < 2000 and p.suffix in (".json", ".py", ".md"):
                print(f"    Content: {p.read_text()[:500]}")
                print()
else:
    print("No artifacts found.")

=== Generated Artifacts ===


## 8. DelegationLoopRunner Direct API

For advanced use cases, you can use `DelegationLoopRunner` directly.
This gives you full control over the configuration, tool registry,
and manager agent setup.

This is what `AgentWorkflow` uses internally.

In [11]:
import tempfile
import yaml
from pathlib import Path

from awp.models.orchestration import (
    DelegationBudget,
    DelegationLoggingConfig,
    DelegationLoopConfig,
    DelegationLoopModels,
    HistoryConfig,
    StallDetectionConfig,
    ValidationConfig,
    WorkerPolicy,
    WorkerPolicyEnforced,
    SandboxEnforcement,
    CodeModeEnforcement,
    RateLimitEnforcement,
)
from awp.models.capabilities import SandboxConfig
from awp.runtime.delegation_loop_runner import DelegationLoopRunner
from awp.runtime.executor_factory import create_executor
from awp.runtime.tools import ToolRegistry

print("Imports successful.")
print("Building DelegationLoopConfig manually...")

# 1. Build the config
config = DelegationLoopConfig(
    manager="agents/manager",
    models=DelegationLoopModels(
        manager=MODEL,
        worker=MODEL,
    ),
    budget=DelegationBudget(
        max_loops=3,
        max_total_workers=10,
        max_total_tokens=80_000,
        max_wall_time=90,
        max_tool_calls=50,
        max_depth=3,
    ),
    worker_policy=WorkerPolicy(
        enforced=WorkerPolicyEnforced(
            sandbox=SandboxEnforcement(type="subprocess"),
            codemode=CodeModeEnforcement(max_tools_per_worker=10),
            rate_limiting=RateLimitEnforcement(),
            forbidden_tools=["shell.execute", "file.write_outside_workspace"],
        ),
        manager_controlled=[
            "instructions", "skills", "tools_allowed",
            "output_contract", "codemode.enabled", "codemode.tool_creation",
        ],
    ),
    termination=StallDetectionConfig(
        enabled=True,
        window=3,
        min_confidence_delta=0.05,
        action="warn_then_stop",
    ),
    validation=ValidationConfig(),
    history=HistoryConfig(
        rolling_summary=True,
        full_results_window=3,
        persist_to_disk=True,
    ),
    logging=DelegationLoggingConfig(
        format="dual",
        persist_artifacts=True,
    ),
)

print(f"Budget: loops={config.budget.max_loops}, workers={config.budget.max_total_workers}, "
      f"tokens={config.budget.max_total_tokens:,}, wall_time={config.budget.max_wall_time}s")
print(f"Models: manager={config.models.manager}, worker={config.models.worker}")

Imports successful.
Building DelegationLoopConfig manually...
Budget: loops=3, workers=10, tokens=80,000, wall_time=90s
Models: manager=openrouter/google/gemini-2.5-flash, worker=openrouter/google/gemini-2.5-flash


In [12]:
# 2. Set up workspace directory with a manager agent
workspace_dir = Path("./output_direct_api").resolve()
workspace_dir.mkdir(parents=True, exist_ok=True)
(workspace_dir / "workspace").mkdir(exist_ok=True)

# Write a manager agent config
manager_dir = workspace_dir / "agents" / "manager"
manager_dir.mkdir(parents=True, exist_ok=True)

# System prompt for the manager
system_prompt = """You are a data analysis manager agent.
You receive a task and delegate it to workers.
Each worker can execute Python code.
Collect worker results and produce a final JSON answer.
"""
(manager_dir / "system_prompt.md").write_text(system_prompt)

# agent.awp.yaml for the manager
agent_config = {
    "awp_agent": "1.0.0",
    "identity": {
        "id": "manager",
        "role": "Data Analysis Manager",
        "version": "1.0.0",
        "description": "Manager agent for direct API demo",
    },
    "runtime": {
        "class_name": "Agent",
        "strategy_folder": "workflow",
    },
    "model": {
        "name": MODEL,
        "temperature": 0.2,
        "max_tokens": 4096,
    },
    "prompt": {
        "system": "system_prompt.md",
    },
    "output": {
        "format": "json",
    },
}
(manager_dir / "agent.awp.yaml").write_text(
    yaml.dump(agent_config, default_flow_style=False, allow_unicode=True)
)

print(f"Workspace: {workspace_dir}")
print(f"Manager agent: {manager_dir}")
print("Files created:")
for p in sorted(workspace_dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(workspace_dir)}")

Workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_direct_api
Manager agent: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/tutorials/output_direct_api/agents/manager
Files created:
  agents/manager/agent.awp.yaml
  agents/manager/system_prompt.md


In [13]:
# 3. Create tool registry and code executor
tool_registry = ToolRegistry(workflow_dir=workspace_dir)
sandbox_cfg = SandboxConfig(
    enabled=True,
    type="subprocess",
)
code_executor = create_executor(sandbox_cfg, working_dir=workspace_dir / "workspace")
tool_registry.set_code_executor(code_executor)

print(f"Tool registry created with executor: {type(code_executor).__name__}")

# 4. Create and run the DelegationLoopRunner
runner = DelegationLoopRunner(
    workflow_dir=workspace_dir,
    config=config,
    tool_registry=tool_registry,
    manager_model=MODEL,
    worker_model=MODEL,
)

task = "Calculate the sum of integers from 1 to 100 and return the result as JSON."
print(f"Running task: {task}")
print()

raw_result = runner.run(task)

print("\n=== Raw Result ===")
print(json.dumps(raw_result, indent=2, default=str))
print()
print(f"Budget used: loops={runner._budget.loops_used}, "
      f"workers={runner._budget.workers_spawned}, "
      f"tokens={runner._budget.tokens_consumed}, "
      f"wall_time={runner._budget.wall_time_elapsed:.1f}s")

INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_03-16-48_0fec7702] depth=0 starting: Calculate the sum of integers from 1 to 100 and return the result as JSON.


INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:awp.runtime.agent:Agent manager LLM error: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


DEBUG:awp.runtime.llm:LLM request: model=google/gemini-2.5-flash, messages=2, tools=0


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:awp.runtime.delegation_loop_runner:Inline manager failed: Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401


Tool registry created with executor: CodeExecutor
Running task: Calculate the sum of integers from 1 to 100 and return the result as JSON.


=== Raw Result ===
{
  "delegation_loop": {
    "error": "Client error '401 Unauthorized' for url 'https://openrouter.ai/api/v1/chat/completions'\nFor more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401",
    "partial_result": {},
    "confidence": 0.0
  }
}

Budget used: loops=1, workers=0, tokens=0, wall_time=0.2s


## 9. Summary

This tutorial covered the two ways to use AWP's delegation loop engine:

### AgentWorkflow (Simple API)
- Import from `awp.data`
- Pass `inputs` (dict, DataFrame, file paths) + `task` (string) + `model`
- Set budget parameters (`max_loops`, `max_wall_time`, `max_total_tokens`, etc.)
- Get back a result dict with `status`, `result`, `artifacts`, `metadata`
- Enable `code_mode=True` and `tool_creation=True` for A3-A4 autonomy

### DelegationLoopRunner (Full Control)
- Build `DelegationLoopConfig` manually with all sub-configs
- Set up workspace directory with manager agent
- Create `ToolRegistry` and code executor
- Call `runner.run(task)` and inspect raw results
- Access `runner._budget` for resource consumption details

### Budget System
The budget system **guarantees termination**. Hard limits (loops, workers, tokens,
wall time, depth) cannot be overridden by the manager. Stall detection adds an
additional safety layer by stopping when no progress is being made.

| Parameter | Default | Description |
|---|---|---|
| `max_loops` | 10 | Max delegation loop iterations |
| `max_total_tokens` | 500,000 | Max total LLM tokens |
| `max_wall_time` | 300 | Max wall time (seconds) |
| `max_tool_calls` | 100 | Max tool invocations |
| `max_total_workers` | 30 | Max worker agents spawned |
| `max_depth` | 5 | Max recursive delegation depth |